# Lofty — YuE GPU Worker for Google Colab

Turns a free T4 GPU into a remote worker for music generation via YuE (Tencent/HKUST).

**YuE** — lyrics-to-song model based on LLaMA. Generates full songs with vocals and accompaniment.

**Free Colab limits:**
- T4 GPU (15 GB VRAM) — uses 4-bit quantization
- Max 2 segments (~60 sec song)
- Generation ~5-8 min per track

---

## Setup (on your PC)

1. `docker compose up -d`
2. `ngrok http 8000`
3. Copy ngrok URL below

**Run in order: Step 1 → 2 → 3 → 4**

---
## Step 1. Configuration and GPU check

In [ ]:
import torch
import requests
import os

# ╔══════════════════════════════════════════════════════╗
# ║  CONFIGURE THESE                                     ║
# ╚══════════════════════════════════════════════════════╝

API_URL = "https://YOUR-NGROK-URL.ngrok-free.dev"
WORKER_API_KEY = "lofty-colab-worker-2026"
POLL_INTERVAL = 5

# ══════════════════════════════════════════════════════

if not torch.cuda.is_available():
    raise RuntimeError("GPU not found! Runtime -> Change runtime type -> T4 GPU")

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
cap = torch.cuda.get_device_capability()
print(f"GPU: {gpu} ({vram:.1f} GB)")
print(f"CUDA: {torch.version.cuda} | Compute: sm_{cap[0]}{cap[1]}")
print(f"PyTorch: {torch.__version__}")

# T4 does NOT support bf16 natively (sm_75), so we use float16
USE_DTYPE = torch.float16
print(f"Compute dtype: {USE_DTYPE}")
print()

try:
    r = requests.get(f"{API_URL}/health", timeout=10,
        headers={"ngrok-skip-browser-warning": "true", "User-Agent": "LoftyWorker/1.0"})
    print(f"API: OK ({r.json()})" if r.status_code == 200 else f"API: {r.status_code}")
except Exception as e:
    print(f"API unavailable: {e}")
    print("Check: docker compose up -d + ngrok http 8000")

---
## Step 2. Install YuE and dependencies

In [ ]:
%%bash
set -e

echo "=== Installing YuE ==="

if [ ! -d "/content/YuE" ]; then
    echo "[1/5] Cloning YuE..."
    git clone --depth 1 https://github.com/multimodal-art-projection/YuE.git /content/YuE
else
    echo "[1/5] YuE already present"
fi

echo "[2/5] Installing YuE dependencies..."
pip install -q torch torchaudio
pip install -q -r /content/YuE/requirements.txt 2>/dev/null

echo "[3/5] Flash Attention 2 (saves VRAM)..."
pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "  flash-attn failed (non-critical, will be slower)"

echo "[4/5] bitsandbytes (4-bit quantization for T4)..."
pip install -q bitsandbytes 2>/dev/null

echo "[5/5] xcodec tokenizer..."
if [ ! -d "/content/YuE/inference/xcodec_mini_infer" ]; then
    cd /content/YuE/inference
    git clone https://huggingface.co/m-a-p/xcodec_mini_infer
else
    echo "  xcodec already present"
fi

which ffmpeg > /dev/null 2>&1 || apt-get -qq install -y ffmpeg > /dev/null 2>&1

echo "=== Done ==="

---
## Step 3. Load models and test

First run downloads ~8 GB of models (Stage 1 in 4-bit, Stage 2, xcodec).

In [ ]:
import sys, os, gc, time, json
import torch
import numpy as np

# Path setup — IMPORTANT for correct YuE imports
YUE_DIR = "/content/YuE"
INFERENCE_DIR = os.path.join(YUE_DIR, "inference")
XCODEC_DIR = os.path.join(INFERENCE_DIR, "xcodec_mini_infer")
DAC_DIR = os.path.join(XCODEC_DIR, "descriptaudiocodec")

# ── Verify xcodec is cloned ──
target_file = os.path.join(XCODEC_DIR, "models", "soundstream_hubert_new.py")
if not os.path.isfile(target_file):
    print(f"ERROR: {target_file} not found!")
    print("Run: !cd /content/YuE/inference && git clone https://huggingface.co/m-a-p/xcodec_mini_infer")
    raise FileNotFoundError(target_file)

# Add paths BEFORE any YuE imports
for p in [XCODEC_DIR, DAC_DIR, INFERENCE_DIR]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

# chdir to INFERENCE_DIR — xcodec config uses relative path "./xcodec_mini_infer/..."
os.chdir(INFERENCE_DIR)
os.environ["YUE_INFERENCE_DIR"] = INFERENCE_DIR

# ── 1. Load Stage 1 (7B, 4-bit) ──
print("[1/4] Stage 1 model (7B, 4-bit NF4)...")
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

try:
    stage1_model = AutoModelForCausalLM.from_pretrained(
        "m-a-p/YuE-s1-7B-anneal-en-cot",
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        attn_implementation="flash_attention_2",
        device_map="auto",
    )
    print("  Stage 1: OK (4-bit, flash-attn)")
except Exception as e:
    print(f"  flash-attn unavailable, loading without it: {e}")
    stage1_model = AutoModelForCausalLM.from_pretrained(
        "m-a-p/YuE-s1-7B-anneal-en-cot",
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    print("  Stage 1: OK (4-bit, no flash-attn)")

stage1_model.eval()
vram_after_s1 = torch.cuda.memory_allocated() / 1024**3
print(f"  VRAM after Stage 1: {vram_after_s1:.1f} GB")

# ── 2. Load tokenizer and codec ──
print("[2/4] Tokenizer and xcodec...")

from codecmanipulator import CodecManipulator
from mmtokenizer import _MMSentencePieceTokenizer
from omegaconf import OmegaConf
from models.soundstream_hubert_new import SoundStream

mmtokenizer = _MMSentencePieceTokenizer(
    os.path.join(INFERENCE_DIR, "mm_tokenizer_v0.2_hf", "tokenizer.model")
)
codectool = CodecManipulator("xcodec", 0, 1)
codectool_stage2 = CodecManipulator("xcodec", 0, 8)

device = torch.device("cuda:0")
xcodec_config = OmegaConf.load(
    os.path.join(XCODEC_DIR, "final_ckpt", "config.yaml")
)
codec_model = eval(xcodec_config.generator.name)(**xcodec_config.generator.config).to(device)
params = torch.load(
    os.path.join(XCODEC_DIR, "final_ckpt", "ckpt_00360000.pth"),
    map_location="cpu", weights_only=False,
)
codec_model.load_state_dict(params["codec_model"])
codec_model.eval()
del params
print("  Tokenizer + xcodec: OK")

# ── 3. Test generation (1 segment, ~30s) ──
print("[3/4] Test generation (30s, 1 segment)...")
import re, random, uuid, copy
from collections import Counter
from einops import rearrange
from transformers import LogitsProcessor, LogitsProcessorList

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class BlockTokenRangeProcessor(LogitsProcessor):
    def __init__(self, start_id, end_id):
        self.blocked = list(range(start_id, end_id))
    def __call__(self, input_ids, scores):
        scores[:, self.blocked] = -float("inf")
        return scores

seed_everything(42)

genres = "upbeat pop electronic female vocal bright"
test_lyrics = "[verse]\nHello world this is a test\nMusic flowing feeling blessed\n\n[chorus]\nLa la la sing along\nThis is our test song\n"

pattern = r"\[(\w+)\](.*?)(?=\[|\Z)"
segments = re.findall(pattern, test_lyrics, re.DOTALL)
lyrics_segments = [f"[{s[0]}]\n{s[1].strip()}\n\n" for s in segments]
full_lyrics = "\n".join(lyrics_segments)

prompt_texts = [f"Generate music from the given lyrics segment by segment.\n[Genre] {genres}\n{full_lyrics}"]
prompt_texts += lyrics_segments

start_of_segment = mmtokenizer.tokenize("[start_of_segment]")
end_of_segment = mmtokenizer.tokenize("[end_of_segment]")

raw_output = None
run_n = min(2, len(prompt_texts))  # 1 segment

t0 = time.time()
try:
    for i in range(1, run_n):
        section_text = prompt_texts[i].replace("[start_of_segment]", "").replace("[end_of_segment]", "")

        head_id = mmtokenizer.tokenize(prompt_texts[0])
        prompt_ids = head_id + start_of_segment + mmtokenizer.tokenize(section_text) + [mmtokenizer.soa] + codectool.sep_ids

        prompt_ids_t = torch.as_tensor(prompt_ids).unsqueeze(0).to(device)
        input_ids = prompt_ids_t

        max_context = 16384 - 3000 - 1
        if input_ids.shape[-1] > max_context:
            input_ids = input_ids[:, -max_context:]

        with torch.no_grad():
            output_seq = stage1_model.generate(
                input_ids=input_ids,
                max_new_tokens=3000,
                min_new_tokens=100,
                do_sample=True,
                top_p=0.93,
                temperature=1.0,
                repetition_penalty=1.1,
                eos_token_id=mmtokenizer.eoa,
                pad_token_id=mmtokenizer.eoa,
                logits_processor=LogitsProcessorList([
                    BlockTokenRangeProcessor(0, 32002),
                    BlockTokenRangeProcessor(32016, 32016),
                ]),
            )
            if output_seq[0][-1].item() != mmtokenizer.eoa:
                tensor_eoa = torch.as_tensor([[mmtokenizer.eoa]]).to(device)
                output_seq = torch.cat((output_seq, tensor_eoa), dim=1)
        raw_output = output_seq

    gen_time = time.time() - t0
    print(f"  Stage 1: {gen_time:.0f}s")

    ids = raw_output[0].cpu().numpy()
    soa_idx = np.where(ids == mmtokenizer.soa)[0].tolist()
    eoa_idx = np.where(ids == mmtokenizer.eoa)[0].tolist()
    print(f"  Segments: {len(soa_idx)} soa, {len(eoa_idx)} eoa")

    if len(soa_idx) == len(eoa_idx) and len(soa_idx) > 0:
        print("  Stage 1: SUCCESS!")
    else:
        print("  Stage 1: ERROR (invalid soa/eoa count)")

except torch.cuda.OutOfMemoryError:
    print("  Stage 1: OOM! Try reducing max_new_tokens or use fewer segments")
except Exception as e:
    print(f"  Stage 1: Error — {e}")

# ── 4. Free VRAM ──
print("[4/4] Clearing Stage 1 from VRAM...")
del stage1_model
if raw_output is not None:
    del raw_output
gc.collect()
torch.cuda.empty_cache()
vram_after = torch.cuda.memory_allocated() / 1024**3
print(f"  VRAM: {vram_after:.1f} GB")

print("\nTest passed! Proceed to Step 4")

---
## Step 4. Worker

Leave this cell running. The worker polls the API and generates music via YuE.

**YuE worker features:**
- Generation ~5-8 min per track (30-60s audio)
- 4-bit quantization Stage 1 (7B) for VRAM savings
- Stage 2 (1B) in float16
- Automatic model offloading between stages
- Vocal + Instrumental → Mix

In [ ]:
import json, time, requests, os, sys, torch, gc, random, re, copy, uuid, io, wave
import numpy as np
import torchaudio
import soundfile as sf
from datetime import datetime
from collections import Counter
from einops import rearrange
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from omegaconf import OmegaConf

# Paths — chdir to INFERENCE_DIR so relative path "./xcodec_mini_infer/..." works
YUE_DIR = "/content/YuE"
INFERENCE_DIR = os.path.join(YUE_DIR, "inference")
XCODEC_DIR = os.path.join(INFERENCE_DIR, "xcodec_mini_infer")
DAC_DIR = os.path.join(XCODEC_DIR, "descriptaudiocodec")
os.chdir(INFERENCE_DIR)

for p in [XCODEC_DIR, DAC_DIR, INFERENCE_DIR]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

HEADERS = {
    "Authorization": f"Bearer {WORKER_API_KEY}",
    "ngrok-skip-browser-warning": "true",
    "User-Agent": "LoftyWorker/1.0",
}

MAX_SEGMENTS = 2
MAX_NEW_TOKENS = 3000

# Language → Stage 1 model
LANG_MODEL_MAP = {
    "en": "m-a-p/YuE-s1-7B-anneal-en-cot",
    "zh": "m-a-p/YuE-s1-7B-anneal-zh-cot",
    "ja": "m-a-p/YuE-s1-7B-anneal-jp-kr-cot",
    "ko": "m-a-p/YuE-s1-7B-anneal-jp-kr-cot",
}
STAGE2_MODEL = "m-a-p/YuE-s2-1B-general"


class BlockTokenRangeProcessor(LogitsProcessor):
    def __init__(self, start_id, end_id):
        self.blocked = list(range(start_id, end_id))
    def __call__(self, input_ids, scores):
        scores[:, self.blocked] = -float("inf")
        return scores


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def split_lyrics(lyrics):
    pattern = r"\[(\w+)\](.*?)(?=\[|\Z)"
    segments = re.findall(pattern, lyrics, re.DOTALL)
    structured = [f"[{s[0]}]\n{s[1].strip()}\n\n" for s in segments]
    return structured if structured else [lyrics.strip()]


def report_progress(job_id, pct):
    try:
        requests.post(f"{API_URL}/api/v1/worker/{job_id}/progress",
            json={"progress": pct}, headers=HEADERS, timeout=5)
    except Exception:
        pass


def check_cancelled(job_id):
    try:
        r = requests.get(f"{API_URL}/api/v1/worker/{job_id}/cancelled",
            headers=HEADERS, timeout=5)
        if r.status_code == 200:
            return r.json().get("cancelled", False)
    except Exception:
        pass
    return False


# ══════════════════════════════════════════════════════
#  Load base components (tokenizer + codec)
# ══════════════════════════════════════════════════════

print("Loading tokenizer and xcodec...")
from codecmanipulator import CodecManipulator
from mmtokenizer import _MMSentencePieceTokenizer
from models.soundstream_hubert_new import SoundStream

mmtokenizer = _MMSentencePieceTokenizer(
    os.path.join(INFERENCE_DIR, "mm_tokenizer_v0.2_hf", "tokenizer.model")
)
codectool = CodecManipulator("xcodec", 0, 1)
codectool_stage2 = CodecManipulator("xcodec", 0, 8)

device = torch.device("cuda:0")
xcodec_config = OmegaConf.load(
    os.path.join(XCODEC_DIR, "final_ckpt", "config.yaml")
)
codec_model = eval(xcodec_config.generator.name)(**xcodec_config.generator.config).to(device)
ckpt = torch.load(
    os.path.join(XCODEC_DIR, "final_ckpt", "ckpt_00360000.pth"),
    map_location="cpu", weights_only=False,
)
codec_model.load_state_dict(ckpt["codec_model"])
codec_model.eval()
del ckpt
print("  OK")

# Model cache (reused between jobs)
_cached_stage1 = None
_cached_stage1_id = None
_cached_stage2 = None


def load_stage1(model_id):
    global _cached_stage1, _cached_stage1_id
    if _cached_stage1 is not None and _cached_stage1_id == model_id:
        return _cached_stage1

    if _cached_stage1 is not None:
        del _cached_stage1
        _cached_stage1 = None
        _cached_stage1_id = None
        torch.cuda.empty_cache()
        gc.collect()

    print(f"  Loading Stage 1: {model_id} (4-bit NF4)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    load_kwargs = {
        "quantization_config": bnb_config,
        "torch_dtype": torch.float16,
        "device_map": "auto",
    }
    try:
        import flash_attn
        load_kwargs["attn_implementation"] = "flash_attention_2"
    except ImportError:
        pass

    model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    model.eval()
    _cached_stage1 = model
    _cached_stage1_id = model_id
    vram = torch.cuda.memory_allocated() / 1024**3
    print(f"  Stage 1 loaded. VRAM: {vram:.1f} GB")
    return model


def offload_stage1():
    global _cached_stage1, _cached_stage1_id
    if _cached_stage1 is not None:
        del _cached_stage1
        _cached_stage1 = None
        _cached_stage1_id = None
    torch.cuda.empty_cache()
    gc.collect()


def load_stage2():
    global _cached_stage2
    if _cached_stage2 is not None:
        return _cached_stage2

    print(f"  Loading Stage 2: {STAGE2_MODEL} (float16)...")
    load_kwargs = {"torch_dtype": torch.float16}
    try:
        import flash_attn
        load_kwargs["attn_implementation"] = "flash_attention_2"
    except ImportError:
        pass

    model = AutoModelForCausalLM.from_pretrained(STAGE2_MODEL, **load_kwargs)
    model.to(device)
    model.eval()
    _cached_stage2 = model
    vram = torch.cuda.memory_allocated() / 1024**3
    print(f"  Stage 2 loaded. VRAM: {vram:.1f} GB")
    return model


def offload_stage2():
    global _cached_stage2
    if _cached_stage2 is not None:
        del _cached_stage2
        _cached_stage2 = None
    torch.cuda.empty_cache()
    gc.collect()


def stage2_generate(model, prompt_npy, batch_size=1):
    """Stage 2: teacher-forcing refinement."""
    codec_ids = codectool.unflatten(prompt_npy, n_quantizer=1)
    codec_ids = codectool.offset_tok_ids(
        codec_ids,
        global_offset=codectool.global_offset,
        codebook_size=codectool.codebook_size,
        num_codebooks=codectool.num_codebooks,
    ).astype(np.int32)

    if batch_size > 1:
        codec_list = [codec_ids[:, i*300:(i+1)*300] for i in range(batch_size)]
        codec_ids = np.concatenate(codec_list, axis=0)
        prompt_ids = np.concatenate([
            np.tile([mmtokenizer.soa, mmtokenizer.stage_1], (batch_size, 1)),
            codec_ids,
            np.tile([mmtokenizer.stage_2], (batch_size, 1)),
        ], axis=1)
    else:
        prompt_ids = np.concatenate([
            np.array([mmtokenizer.soa, mmtokenizer.stage_1]),
            codec_ids.flatten(),
            np.array([mmtokenizer.stage_2]),
        ]).astype(np.int32)
        prompt_ids = prompt_ids[np.newaxis, ...]

    codec_ids_t = torch.as_tensor(codec_ids).to(device)
    prompt_ids_t = torch.as_tensor(prompt_ids).to(device)
    len_prompt = prompt_ids_t.shape[-1]

    block_list = LogitsProcessorList([
        BlockTokenRangeProcessor(0, 46358),
        BlockTokenRangeProcessor(53526, mmtokenizer.vocab_size),
    ])

    for frame_idx in range(codec_ids_t.shape[1]):
        cb0 = codec_ids_t[:, frame_idx:frame_idx+1]
        prompt_ids_t = torch.cat([prompt_ids_t, cb0], dim=1)
        with torch.no_grad():
            out = model.generate(
                input_ids=prompt_ids_t,
                min_new_tokens=7, max_new_tokens=7,
                eos_token_id=mmtokenizer.eoa,
                pad_token_id=mmtokenizer.eoa,
                logits_processor=block_list,
            )
        prompt_ids_t = out

    if batch_size > 1:
        output = prompt_ids_t.cpu().numpy()[:, len_prompt:]
        output = np.concatenate([output[i] for i in range(batch_size)], axis=0)
    else:
        output = prompt_ids_t[0].cpu().numpy()[len_prompt:]
    return output


def stage2_inference(stage1_outputs, batch_size=1):
    """Run Stage 2 on all stage1 output tracks."""
    results = []
    for npy_data in stage1_outputs:
        prompt = npy_data.astype(np.int32)
        output_duration = prompt.shape[-1] // 50 // 6 * 6
        num_batch = output_duration // 6

        if num_batch <= batch_size:
            output = stage2_generate(_cached_stage2, prompt[:, :output_duration*50], batch_size=num_batch)
        else:
            segments = []
            num_segs = (num_batch // batch_size) + (1 if num_batch % batch_size != 0 else 0)
            for seg in range(num_segs):
                start = seg * batch_size * 300
                end = min((seg + 1) * batch_size * 300, output_duration * 50)
                cur_bs = batch_size if seg != num_segs - 1 or num_batch % batch_size == 0 else num_batch % batch_size
                segments.append(stage2_generate(_cached_stage2, prompt[:, start:end], batch_size=cur_bs))
            output = np.concatenate(segments, axis=0)

        if output_duration * 50 != prompt.shape[-1]:
            ending = stage2_generate(_cached_stage2, prompt[:, output_duration*50:], batch_size=1)
            output = np.concatenate([output, ending], axis=0)

        output = codectool_stage2.ids2npy(output)

        # Fix invalid codes
        fixed = copy.deepcopy(output)
        for i, line in enumerate(output):
            for j, elem in enumerate(line):
                if elem < 0 or elem > 1023:
                    counter = Counter(line)
                    most_freq = sorted(counter.items(), key=lambda x: x[1], reverse=True)[0][0]
                    fixed[i, j] = most_freq

        results.append(fixed)
    return results


def generate_music(job):
    """Full YuE generation pipeline."""
    job_id = job["job_id"]
    params = job.get("generation_params", {}) or {}
    lyrics = job.get("lyrics", "") or ""
    prompt = str(job.get("prompt", "") or "")
    language = params.get("language", "en") or "en"
    temperature = params.get("temperature", 1.0)
    top_p = params.get("top_p", 0.93)
    repetition_penalty = params.get("repetition_penalty", 1.1)
    max_new_tokens = params.get("max_new_tokens", MAX_NEW_TOKENS)
    num_segments = min(params.get("num_segments", 1), MAX_SEGMENTS)
    seed = params.get("seed", 42)
    if seed is not None and int(seed) >= 0:
        seed_everything(int(seed))
    else:
        seed_everything(random.randint(0, 999999))

    report_progress(job_id, 5)

    stage1_model_id = LANG_MODEL_MAP.get(language, LANG_MODEL_MAP["en"])

    lyrics_segments = split_lyrics(lyrics)
    full_lyrics = "\n".join(lyrics_segments)
    genres = prompt.strip()

    prompt_texts = [
        f"Generate music from the given lyrics segment by segment.\n[Genre] {genres}\n{full_lyrics}"
    ]
    prompt_texts += lyrics_segments

    start_of_segment = mmtokenizer.tokenize("[start_of_segment]")
    end_of_segment = mmtokenizer.tokenize("[end_of_segment]")

    # ── Stage 1 ──
    print(f"  Stage 1: generating tokens ({num_segments} seg)...")
    stage1 = load_stage1(stage1_model_id)
    report_progress(job_id, 10)

    run_n = min(num_segments + 1, len(prompt_texts))
    raw_output = None

    try:
        for i in range(1, run_n):
            if check_cancelled(job_id):
                raise KeyboardInterrupt("Cancelled")

            section_text = prompt_texts[i].replace("[start_of_segment]", "").replace("[end_of_segment]", "")

            if i == 1:
                head_id = mmtokenizer.tokenize(prompt_texts[0])
                p_ids = head_id + start_of_segment + mmtokenizer.tokenize(section_text) + [mmtokenizer.soa] + codectool.sep_ids
            else:
                p_ids = end_of_segment + start_of_segment + mmtokenizer.tokenize(section_text) + [mmtokenizer.soa] + codectool.sep_ids

            p_ids_t = torch.as_tensor(p_ids).unsqueeze(0).to(device)
            input_ids = torch.cat([raw_output, p_ids_t], dim=1) if i > 1 else p_ids_t

            max_context = 16384 - max_new_tokens - 1
            if input_ids.shape[-1] > max_context:
                input_ids = input_ids[:, -max_context:]

            with torch.no_grad():
                output_seq = stage1.generate(
                    input_ids=input_ids,
                    max_new_tokens=max_new_tokens,
                    min_new_tokens=100,
                    do_sample=True,
                    top_p=top_p,
                    temperature=temperature,
                    repetition_penalty=repetition_penalty,
                    eos_token_id=mmtokenizer.eoa,
                    pad_token_id=mmtokenizer.eoa,
                    logits_processor=LogitsProcessorList([
                        BlockTokenRangeProcessor(0, 32002),
                        BlockTokenRangeProcessor(32016, 32016),
                    ]),
                )
                if output_seq[0][-1].item() != mmtokenizer.eoa:
                    tensor_eoa = torch.as_tensor([[mmtokenizer.eoa]]).to(device)
                    output_seq = torch.cat((output_seq, tensor_eoa), dim=1)

            if i > 1:
                raw_output = torch.cat([raw_output, p_ids_t, output_seq[:, input_ids.shape[-1]:]], dim=1)
            else:
                raw_output = output_seq

            pct = 10 + int((i / (run_n - 1)) * 35)
            report_progress(job_id, min(pct, 45))
            print(f"    Segment {i}/{run_n-1} done")

    except torch.cuda.OutOfMemoryError:
        offload_stage1()
        raise RuntimeError("OOM during Stage 1 generation. Try fewer segments or shorter lyrics.")

    report_progress(job_id, 50)

    # Extract vocal and instrumental
    ids = raw_output[0].cpu().numpy()
    soa_idx = np.where(ids == mmtokenizer.soa)[0].tolist()
    eoa_idx = np.where(ids == mmtokenizer.eoa)[0].tolist()

    if len(soa_idx) != len(eoa_idx):
        raise RuntimeError(f"Invalid soa/eoa: {len(soa_idx)} vs {len(eoa_idx)}")

    vocals_list, inst_list = [], []
    for idx in range(len(soa_idx)):
        codec_ids = ids[soa_idx[idx]+1:eoa_idx[idx]]
        if len(codec_ids) > 0 and codec_ids[0] == 32016:
            codec_ids = codec_ids[1:]
        codec_ids = codec_ids[:2*(codec_ids.shape[0]//2)]
        vocals_list.append(codectool.ids2npy(rearrange(codec_ids, "(n b) -> b n", b=2)[0]))
        inst_list.append(codectool.ids2npy(rearrange(codec_ids, "(n b) -> b n", b=2)[1]))

    vocals_npy = np.concatenate(vocals_list, axis=1)
    inst_npy = np.concatenate(inst_list, axis=1)

    del raw_output
    torch.cuda.empty_cache()

    # ── Offload Stage 1, load Stage 2 ──
    offload_stage1()
    report_progress(job_id, 55)

    print("  Stage 2: refining tokens...")
    load_stage2()
    report_progress(job_id, 60)

    if check_cancelled(job_id):
        offload_stage2()
        raise KeyboardInterrupt("Cancelled")

    stage2_results = stage2_inference([vocals_npy, inst_npy], batch_size=1)
    report_progress(job_id, 80)

    offload_stage2()

    # ── Decode audio ──
    print("  Decoding audio...")
    tracks = []
    for result_npy in stage2_results:
        with torch.no_grad():
            decoded = codec_model.decode(
                torch.as_tensor(result_npy.astype(np.int16), dtype=torch.long)
                .unsqueeze(0).permute(1, 0, 2).to(device)
            )
        tracks.append(decoded.cpu().squeeze(0))

    report_progress(job_id, 90)

    # Mix vocal + instrumental
    min_len = min(tracks[0].shape[-1], tracks[1].shape[-1])
    mixed = (tracks[0][..., :min_len] + tracks[1][..., :min_len]) / 2.0
    mixed_np = mixed.numpy()

    if mixed_np.ndim == 1:
        mixed_np = mixed_np[np.newaxis, :]

    peak = max(abs(mixed_np.max()), abs(mixed_np.min()))
    if peak > 0:
        mixed_np = mixed_np / peak

    audio_int16 = (mixed_np * 32767).astype(np.int16)
    n_channels = min(mixed_np.shape[0], 2)

    if n_channels == 2:
        interleaved = np.empty(audio_int16.shape[1] * 2, dtype=np.int16)
        interleaved[0::2] = audio_int16[0]
        interleaved[1::2] = audio_int16[1]
    else:
        interleaved = audio_int16[0]

    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(n_channels)
        wf.setsampwidth(2)
        wf.setframerate(16000)
        wf.writeframes(interleaved.tobytes())

    wav_bytes = buf.getvalue()
    actual_duration = mixed_np.shape[-1] / 16000

    report_progress(job_id, 95)
    return wav_bytes, 16000, actual_duration


# ══════════════════════════════════════════════════════
#  Main loop
# ══════════════════════════════════════════════════════
print("=" * 60)
print(f"  Lofty YuE Worker")
print(f"  API:    {API_URL}")
print(f"  Model:  YuE (Stage1 7B 4-bit + Stage2 1B)")
print(f"  Max:    {MAX_SEGMENTS} seg, {MAX_NEW_TOKENS} tokens/seg")
print(f"  Poll:   every {POLL_INTERVAL} sec")
print(f"  Langs:  en, zh, ja, ko")
print("=" * 60)
print("\nWaiting for jobs...")

jobs_done = 0
jobs_failed = 0
total_audio = 0.0

while True:
    try:
        resp = requests.get(
            f"{API_URL}/api/v1/worker/next-job?engine=yue",
            headers=HEADERS, timeout=15,
        )

        if resp.status_code == 200:
            job = resp.json()
            job_id = job["job_id"]

            print(f"\n[{datetime.now().strftime('%H:%M:%S')}] "
                  f"Generating: {str(job_id)[:8]}...")
            print(f"  Prompt: {job['prompt'][:60]}")
            print(f"  Duration: {job.get('duration_seconds', 30)} sec")
            if job.get("lyrics", "").strip():
                print(f"  Lyrics: {len(job['lyrics'])} chars")

            try:
                t0 = time.time()
                wav_bytes, sample_rate, duration = generate_music(job)
                gen_time = time.time() - t0

                upload_resp = requests.post(
                    f"{API_URL}/api/v1/worker/{job_id}/result",
                    headers=HEADERS,
                    files={"audio_file": ("track.wav", wav_bytes, "audio/wav")},
                    data={
                        "status": "completed",
                        "duration": str(duration),
                        "sample_rate": str(sample_rate),
                        "format": "wav",
                    },
                    timeout=60,
                )

                if upload_resp.status_code != 200:
                    raise RuntimeError(f"Upload error: {upload_resp.status_code}")

                jobs_done += 1
                total_audio += duration
                print(f"  Done in {gen_time:.0f}s ({duration:.0f}s audio)")
                print(f"  Total: {jobs_done} tracks, {total_audio:.0f}s audio")

            except KeyboardInterrupt:
                print(f"  [!] Cancelled")
                try:
                    requests.post(
                        f"{API_URL}/api/v1/worker/{job_id}/result",
                        headers=HEADERS,
                        data={"status": "cancelled"},
                        timeout=10)
                except Exception:
                    pass

            except Exception as e:
                jobs_failed += 1
                print(f"  Error: {e}")
                try:
                    requests.post(
                        f"{API_URL}/api/v1/worker/{job_id}/result",
                        headers=HEADERS,
                        data={"status": "failed", "error_message": str(e)[:500]},
                        timeout=10)
                except Exception:
                    pass

            torch.cuda.empty_cache()
            gc.collect()
            continue

        time.sleep(POLL_INTERVAL)

    except KeyboardInterrupt:
        print(f"\nStopped. Tracks: {jobs_done}, Errors: {jobs_failed}")
        break

    except Exception as e:
        print(f"  Loop error: {e}")
        time.sleep(5)

---
## Utilities

### GPU Status

In [ ]:
import torch, psutil
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"RAM:  {psutil.virtual_memory().used/1024**3:.1f} / {psutil.virtual_memory().total/1024**3:.1f} GB")
print(f"Disk: {psutil.disk_usage('/').used/1024**3:.1f} / {psutil.disk_usage('/').total/1024**3:.1f} GB")

### Clear VRAM

In [ ]:
import torch, gc
for var in ['_cached_stage1', '_cached_stage2']:
    if var in globals() and globals()[var] is not None:
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
print(f"Cleared. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")